# Semantic Kernel 

ಈ ಕೋಡ್ ಸ್ಯಾಂಪಲ್‌ನಲ್ಲಿ, ನೀವು [Semantic Kernel](https://aka.ms/ai-agents-beginners/semantic-kernel) AI ಫ್ರೇಮ್ವರ್ಕ್ ಅನ್ನು ಬಳಸಿಕೊಂಡು ಸರಳ ಏಜೆಂಟ್ ಅನ್ನು ಸೃಷ್ಟಿಸುತ್ತೀರಿ.

ಈ ಸ್ಯಾಂಪಲ್‌ನ ಉದ್ದೇಶವು ಇದೆಯೆಂದರೆ ವಿವಿಧ ಏಜೆಂಟಿಕ್ ಮಾದರಿಗಳನ್ನು ಅನುಷ್ಠಾನಗೊಳಿಸುವ ಸಂದರ್ಭದಲ್ಲಿ ನಂತರ ಅನೇಕ ಕೋಡ್ ಸ್ಯಾಂಪಲ್ಸ್‌ನಲ್ಲಿ ಅನ್ವಯಿಸುವ ಹಂತಗಳನ್ನು ಪ್ರದರ್ಶಿಸುವುದು.


## ಅಗತ್ಯವಿರುವ ಪೈಠನ್ ಪ್ಯಾಕೇಜ್‌ಗಳನ್ನು ಆಮದುಮಾಡಿ


In [ ]:
import json
import os 

from typing import Annotated

from dotenv import load_dotenv

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

## ಕ್ಲೈಂಟ್ ರಚನೆ

ಈ ಉದಾಹರಣೆಯಲ್ಲಿ, ನಾವು LLM ಅನ್ನು ಪ್ರವೇಶಿಸಲು [GitHub ಮಾದರಿಗಳನ್ನು](https://aka.ms/ai-agents-beginners/github-models) ಬಳಸುತ್ತೇವೆ.

`ai_model_id` ಅನ್ನು `gpt-4o-mini` ಎಂದು ಹೊಂದಿಸಲಾಗಿದೆ. ವಿಭಿನ್ನ ಫಲಿತಾಂಶಗಳನ್ನು ಗಮನಿಸಲು GitHub ಮಾದರಿಗಳ ಮಾರುಕಟ್ಟೆಯಲ್ಲಿ ಲಭ್ಯವಿರುವ ಇನ್ನೊಂದು ಮಾದರಿಗೆ ಬದಲಾಯಿಸಲು ಪ್ರಯತ್ನಿಸಿ.

GitHub ಮಾದರಿಗಳ `base_url`ಿಗೆ ಬೇಕಾದ `Azure Inference SDK` ಅನ್ನು ಬಳಸಲು, ನಾವು Semantic Kernel ಒಳಗಿನ `OpenAIChatCompletion` ಸಂಪರ್ಕকারী ಅನ್ನು ಬಳಸುತ್ತೇವೆ. ನೀವು Semantic Kernel ಅನ್ನು ಬೇರೆ ಮಾದರಿ ಒದಗಿಸುವವರೊಂದಿಗೆ ಬಳಸಲು ಸಾಧ್ಯವಾಗಿಸುವ ಇನ್ನೇನು [ಲಭ್ಯವಿರುವ ಸಂಪರ್ಕಕಾರಿಗಳು](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion) ಕೂಡ ಇವೆ.


In [ ]:
import random   

# Define a sample plugin for the sample

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""

    def __init__(self):
        # List of vacation destinations
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        self.last_destination = None

    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        destination = random.choice(available_destinations)

        # Update the last destination
        self.last_destination = destination

        return destination

In [ ]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    base_url="https://models.inference.ai.azure.com/",
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
)

## ಏಜೆಂಟ್ ರಚನೆ

ಇಲ್ಲಿ, ನಾವು `TravelAgent` ಎಂಬ ಏಜೆಂಟ್ ಅನ್ನು ರಚಿಸುತ್ತೇವೆ.

ಈ ಉದಾಹರಣೆಯಲ್ಲಿ, ನಾವು 아주 ಸರಳ ಸೂಚನೆಗಳನ್ನು ಬಳಸುತ್ತೇವೆ. ಏಜೆಂಟ್ ನ ನಡೆ ಹೇಗೆ ಬದಲಾಗುತ್ತದೆ ಎಂಬುದನ್ನು ಗಮನಿಸಲು ಈ ಸೂಚನೆಗಳನ್ನು ನೀವು ಬದಲಾಯಿಸಬಹುದು.


In [ ]:
agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
)

## ಏಜೆಂಟ್ಗಳನ್ನು ಚಾಲನೆ ಮಾಡುವುದು

ಈಗ ನಾವು `ChatHistory` ಅನ್ನು ಸೆಟ್ ಅಪ್ ಮಾಡಿ, ಅದರಲ್ಲಿ `system_message` ಅನ್ನು ಸೇರಿಸುವ ಮೂಲಕ ಏಜೆಂಟ್ ಅನ್ನು ಕಾರ್ಯಗತಗೊಳಿಸಬಹುದು. ನಾವು ಹಿಂದಿನದಾಗಿ ಸ್ಥಾಪಿಸಿರುವ `AGENT_INSTRUCTIONS` ಅನ್ನು ಬಳಸುತ್ತೇವೆ.

ಇವುಗಳನ್ನು ಸೆಟ್ ಅಪ್ ಮಾಡಿದ ನಂತರ, ನಾವು `user_inputs` ರಚಿಸುತ್ತೇವೆ, ಅದು ಬಳಕೆದಾರರು ಏಜೆಂಟ್ ಗೆ ಏನನ್ನು ಕಳುಹಿಸುತ್ತಿದ್ದಾರೆ ಎಂಬುದನ್ನು ಪ್ರತಿನಿಧಿಸುತ್ತದೆ. ಈ ಉದಾಹರಣೆಯಲ್ಲಿ, ಸಂದೇಶವು `ನನಗೆ ಒಬ್ಬ ಸೂರ್ಯಕಿರಣಯುಕ್ತ ರಜೆ ಯೋಜನೆ ಮಾಡು` ಎಂದು ಸೆಟ್ ಆಗಿದೆ.

ನೀವು ಈ ಸಂದೇಶವನ್ನು ತಿದ್ದುಪಡಿ ಮಾಡಿ, ಏಜೆಂಟ್ ವಿಭಿನ್ನವಾಗಿ ಪ್ರತಿಕ್ರಿಯಿಸುವುದನ್ನು ಗಮನಿಸಬಹುದು.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]

async def main():
    thread: ChatHistoryAgentThread | None = None

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # Buffer to reconstruct streaming function call
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # Accumulate arguments (streamed in chunks)
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # Finalize any pending function call before showing result
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # leave as raw string

                        function_calls.append(f"Calling function: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\nFunction Result:\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಒಪ್ಪಿಗೆಯ ನಿರಾಕರಣೆ**:  
ಈ ದಸ್ತಾವೇಜನ್ನು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ಶುದ್ಧತೆಯೊಂದಿಗೇ ಕೆಲಸ ಮಾಡುವ ಪ್ರಯತ್ನದಲ್ಲಿದ್ದರೂ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳಾಗಬಹುದು ಅಥವಾ ತಪ್ಪುಗಳಾಗಿರಬಹುದು ಎಂದು ದಯವಿಟ್ಟು ಗಮನಿಸಿ. ಮೂಲದ ದಸ್ತಾವೇಜಿನ ಸ್ಥಳೀಯ ಭಾಷೆಯ ಆವೃತ್ತಿಯನ್ನು ಅಧಿಕೃತ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಮುಖ್ಯ ಮಾಹಿತಿಗಾಗಿ, ಪರಿಣತಿ ಹೊಂದಿದ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಮಾಡಿಕೋಲೆ ಅಥವಾ ತಪ್ಪಾಗುಮನಗಳಿಗೆ ನಾವು ಜವಾಬ್ದಾರರಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
